# Pipeline arXiv — amostra balanceada (15 categorias × 100 = 1500 linhas)

Esta pipeline lê o dataset de metadados do arXiv (`arxiv-metadata-oai-snapshot.json`, ~5,3 GB, um JSON por linha) de forma **streaming** (linha a linha, sem carregar tudo na RAM) e gera uma amostra com **100 papers por categoria** para as **15 maiores categorias** do projeto.

**Resultado:** `arxiv_amostra_1500.json` (1500 registros) — baixado localmente para o seu PC.

> Nomes de categoria já corrigidos: `cond.mat.mtrl-sci`→`cond-mat.mtrl-sci` e `cond-matstat-mech`→`cond-mat.stat-mech`.

## 1. Configuração

In [1]:
# As 15 maiores categorias do projeto (nomes corrigidos)
TARGET_CATS = [
    "cs.LG", "hep-ph", "cs.CV", "cs.AI", "hep-th",
    "quant-ph", "gr-qc", "cs.CL", "cond-mat.mtrl-sci", "astro-ph",
    "cond-mat.mes-hall", "math-ph", "math.MP", "cond-mat.str-el", "cond-mat.stat-mech",
]

N_POR_CAT = 100                       # linhas por categoria  -> 15 * 100 = 1500
CAMINHO   = "arxiv-metadata-oai-snapshot.json"   # ajustado na próxima seção
SAIDA     = "arxiv_amostra_1500.json"

assert len(TARGET_CATS) == 15, "Esperado 15 categorias"
print(f"{len(TARGET_CATS)} categorias x {N_POR_CAT} = {len(TARGET_CATS)*N_POR_CAT} linhas")

15 categorias x 100 = 1500 linhas


## 2. Obter o dataset

Escolha **UMA** das opções abaixo conforme onde o arquivo está.

### Opção A — Google Drive (se você já subiu o `.json` para o Drive)

In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')

# Pasta do projeto no Drive
BASE = '/content/drive/MyDrive/projetoIA-EquipeLoremIpsum'

# Entrada: o .json está na pasta do projeto
CAMINHO = os.path.join(BASE, 'source-arxiv', 'arxiv-metadata-oai-snapshot.json')

# Saída: salva dentro da pasta 'pipelines' (mesma do notebook)
SAIDA = os.path.join(BASE, 'pipelines', 'arxiv_amostra_1500.json')

print('Entrada existe?', os.path.exists(CAMINHO),
      '| Tamanho (GB):', round(os.path.getsize(CAMINHO)/1e9, 2) if os.path.exists(CAMINHO) else '-')
print('Pasta de saida existe?', os.path.isdir(os.path.dirname(SAIDA)))

Mounted at /content/drive
Entrada existe? True | Tamanho (GB): 5.34
Pasta de saida existe? True


## 3. Pipeline de filtragem (streaming)

Estratégia de atribuição: cada paper entra em **um único** bucket. Preferimos a categoria **primária** (primeira da lista) quando ela é um alvo que ainda precisa de linhas; senão, usamos qualquer categoria-alvo presente que ainda esteja faltando. Assim não há linhas duplicadas e cada bucket fecha exatamente em 100. O loop para assim que as 15 categorias estão completas.

In [3]:
import json
from collections import defaultdict

target_set = set(TARGET_CATS)
buckets    = defaultdict(list)
restantes  = set(TARGET_CATS)        # categorias que ainda não atingiram N_POR_CAT
lidos = 0

with open(CAMINHO, encoding='utf-8') as f:
    for linha in f:
        if not restantes:            # todas completas -> para cedo
            break
        lidos += 1
        p = json.loads(linha)
        cats = p['categories'].split()
        if not cats:
            continue

        # escolhe o bucket: primária primeiro, depois qualquer alvo faltando
        escolha = cats[0] if cats[0] in restantes else None
        if escolha is None:
            for c in cats:
                if c in restantes:
                    escolha = c
                    break
        if escolha is None:
            continue

        buckets[escolha].append({
            'id': p['id'],
            'title': (p.get('title') or '').strip(),
            'abstract': (p.get('abstract') or '').strip(),
            'categories': p['categories'],
            'primary_category': cats[0],
            'assigned_category': escolha,
            'authors': p.get('authors'),
            'update_date': p.get('update_date'),
        })
        if len(buckets[escolha]) >= N_POR_CAT:
            restantes.discard(escolha)

print(f'Linhas lidas do dataset: {lidos:,}')
print(f'Categorias completas: {len(TARGET_CATS) - len(restantes)}/{len(TARGET_CATS)}')
if restantes:
    print('ATENÇÃO — não atingiram 100:', restantes)

Linhas lidas do dataset: 83,376
Categorias completas: 15/15


## 4. Montar o DataFrame e validar

In [4]:
import pandas as pd

linhas = [row for cat in TARGET_CATS for row in buckets[cat]]
df = pd.DataFrame(linhas)

print('Shape:', df.shape)
print('\nLinhas por categoria (assigned_category):')
print(df['assigned_category'].value_counts().reindex(TARGET_CATS))
print('\nIDs duplicados:', df['id'].duplicated().sum())
df.head()

Shape: (1500, 8)

Linhas por categoria (assigned_category):
assigned_category
cs.LG                 100
hep-ph                100
cs.CV                 100
cs.AI                 100
hep-th                100
quant-ph              100
gr-qc                 100
cs.CL                 100
cond-mat.mtrl-sci     100
astro-ph              100
cond-mat.mes-hall     100
math-ph               100
math.MP               100
cond-mat.str-el       100
cond-mat.stat-mech    100
Name: count, dtype: int64

IDs duplicados: 0


,id,title,abstract,categories,primary_category,assigned_category,authors,update_date
0,0704.0671,Learning from compressed observations,The problem of statistical learning is to cons...,cs.IT cs.LG math.IT,cs.IT,cs.LG,Maxim Raginsky,2016-11-15
1,0704.0954,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the communic...",cs.IT cs.LG math.IT,cs.IT,cs.LG,Soummya Kar and Jose M. F. Moura,2009-11-13
2,0704.1020,The on-line shortest path problem under partia...,The on-line shortest path problem is considere...,cs.LG cs.SC,cs.LG,cs.LG,"Andras Gyorgy, Tamas Linder, Gabor Lugosi, Gyo...",2007-05-23
3,0704.1028,A neural network approach to ordinal regression,Ordinal regression is an important type of lea...,cs.LG cs.AI cs.NE,cs.LG,cs.LG,Jianlin Cheng,2007-05-23
4,0704.1274,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close rel...,cs.LG,cs.LG,cs.LG,David H. Wolpert and Dev G. Rajnarayan,2011-11-09


## 5. Salvar como JSON Lines e baixar para o PC

Salva no **mesmo formato do arquivo-fonte**: JSON Lines (um objeto JSON por linha, `lines=True`, UTF-8 preservando acentos) e dispara o download para a sua máquina.

In [5]:
# Salva no mesmo formato do arquivo-fonte: JSON Lines (um objeto JSON por linha)
df.to_json(SAIDA, orient='records', lines=True, force_ascii=False)
print('Salvo:', SAIDA, '|', len(df), 'registros (JSON Lines)')

# baixa para a sua máquina (Colab)
try:
    from google.colab import files
    files.download(SAIDA)
except Exception as e:
    print('Download automático indisponível (rodando fora do Colab?):', e)

Salvo: /content/drive/MyDrive/projetoIA-EquipeLoremIpsum/pipelines/arxiv_amostra_1500.json | 1500 registros (JSON Lines)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>